# Image appraisal in geoelectrics
Image appraisal in geophysics refers to the evaluation of the quality, reliability, and accuracy of inverse models derived from geophysical data. It involves assessing factors such as resolution, sensitivity, and model parameter uncertainties to ensure the validity of subsurface interpretations. 

This tutorial should give an overview of basic image appraisal methods in geoeletrical monitoring that could provide helpful tools for your future synthetic studies as well as field applications.

### Content of this tutorial:

**1) Returning and plotting sensitivities of single configurations**

**2) Visualizing the coverage of a dataset**

**3) Assessing the Depth of Investigation** (Oldenburg and Li, 1999)


In [1]:
import matplotlib.pyplot as plt
import numpy as np

import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import ert

## 1) Assessing the sensitivity of single configurations

We will start this tutorial by **assessing sensitivities of single four-point arrays**.  

You can either use a dataset of your choice or utilize the example dataset provided in the GIT repo. As input for this exercise, we only need an inverse model + corresponding mesh. Please start by importing your dataset or inverse model + mesh:

##### **Import your data here:**

Based on the provided inversion result, we now want to plot the sensitivity of single four-point arrays, as seen in [this example on the pyGIMLi website](https://www.pygimli.org/_examples_auto/3_ert/plot_05_ert_4_point_sensitivities.html).

#### 1a) Start by getting the Jacobian matrix (and visualize its dimensions). After doing so, pick a single configuration and plot its normalized, logarithmic sensitivity distribution. Optionally, mark the current and potential electrodes used for the particular configuration in question.

##### **Result:**

#### 1b) Based on the already implemented code, get the four point array with the highest and lowest geometric factor K and plot its normalized sensitivity distribution on the model domain.

##### **Result:**

## 2) Displaying the coverage of a dataset

After introducing the sensitivity of single four-point configurations, we now want to take a look at another quality assessment parameter often used in geoelectrical imaging - the coverage. Coverage is defined as the **spatial distribution of the sensitivity of all measurements across the model space**. It indicates which regions are well covered and which areas lack data influence, thereby **helping to evaluate model reliability**.

#### 2a) Based on the previous results, calculate the coverage for the given example dataset using pyGIMLi's `coverage()` method of the ERTManager.

##### **Result:**

This plot nicely visualizes the depth dependency of the coverage in the model space. Going one step further, we can make use of the coverage to cut off those parts of the inverse model that correspond with areas of low coverage.

#### 2b) Define a function that masks the inverse model based on a given quality evaluation parameter. The function should allow to manually change the cutoff value. Utilize the coverage as input and plot the inversion result with the defined masking function.
##### _Hint: Use the variable coverage within the `mgr.showResult()` method to manually adjust the cutoff in the plot_.

##### **Result:**

This is a great and versatile tool to visually appraise the quality of an inverse model and, since any model space evaluation parameter can be inserted, it can be utilized for any parameter of choice!

Next up, we want to take a look into the evaluation of the image quality at a specific location with depth. Let's say, you want to get an evaluation of the image quality at a specific point along your survey to consider whether additional measurements would be helpful. By using pyGIMLi's interpolation function, we can build a tool that enables exactly that! 

#### 2c) Starting with the code snipped below, build a tool to assess the coverage changes with depth at a specific x-coordinate along the profile. _Hint: Use pyGIMLi's interpolate function!._

##### **Result:**

## Takeaway messages
- We visualized the **sensitivities of single four point arrays** on the model space and underlined **differences in the distribution** based on the electrode and dipole spacing. This helps to get a feeling of which configurations provide information on which part of the model space. 
---
- We introduced the coverage as tool to assess the  **spatial distribution of the sensitivity of all measurements across the model space**, which allows to **objectively evaluate the reliability** in different parts of the model.
---
- We build a tool that allows us to **cut out areas** from the result plot that hold **low coverage**. However, this **tool is method-independent** and can therefore also be utilized with other quality evaluation parameters, such as the Depth of Investigation (DOI) and the model resolution. 
---
- We constructed a tool that allows us to assess the quality in a 1D column anywhere on the model. This way, we selectively evaluate the model quality for a specific region on the model space. 

---
## **Advanced section**
---

We presented powerful tools to assess the quality of inversion results, both locally as well as on the whole model space. However, the coverage and sensitivity provide only limited capacities as they do not consider regularization and linear (in)dependencies of different measurements.

For this purpose, we introduce another tool for the appraisal of geoelectrical inversion results - the **Depth of Investigation (DOI)** (from Oldenburg, D. W., & Li, Y. (1999). Estimating depth of investigation in DC resistivity and IP surveys. GEOPHYSICS. Society of Exploration Geophysicists. https://doi.org/10.1190/1.1444545).

DOI generally refers to the depth of the model space, **"below which surface data are insensitive to the value of the physical property"**, in our case to the electrical resistivity. Thus, the **interpretation of features below this depth should be handled with care**, since they might not be geologically feasible.

If we consider two inversion results that are carried out with **two different, homogeneous reference models** (e.g., two homogeneous reference models with $\rho (m_{1,r}) = $ 100 $\Omega m$ and b) $\rho (m_{1,r})$ = 500 $\Omega m$) and their inverse models $m_1$ and $m_2$, the DOI is computed by:
$$
R(x, z)=\frac{m_{1}(x, z)-m_{2}(x, z)}{m_{1 r}-m_{2 r}}
$$

In theory, the quality assessment parameter **R will approach 0** in regions of the model, where the **data constrain the model**, which means that the inversion does not significantly depend on the defined reference model. Thus, the inverse model provides **reliable information in those model areas**. Vice versa, if the inversion result shows values similar to the reference model, the credibility of the corresponding model regions should be low. 

Thus, the DOI provides a valuable tool to **assess the credibility of the inversion result** over the whole model domain.

#### a) Define a function to compute the DOI (Oldenburg and Li, 1999) for the previous inversion result. Use the following code snippet to set the reference models:

In [ ]:
# Definition of reference models for DOI computation:
data = pg.load("YOUR DATASET")
mgr = ert.ERTManager(data)

def run_inv(referencemodel):
    # Set reference model as startmodel
    mgr.inv.inv.setModel(referencemodel)

    # Set reference model
    rm = mgr.inv.fop.regionManager()
    rm.setConstraintType(10) # 0 = reference model, 1 = first-order smoothing (default), 10 = both
    rm.fillConstraints(mgr.inv.fop.constraints())
    mgr.inv.inv.setReferenceModel(referencemodel)
    model = mgr.inv.inv.run()
    return model

##### **Result:**

#### b) Use the previously defined cutoff function to plot the computed DOI value. Do you have to change something in the function?

##### **Result:**